# PhenoBench Multiclass — full frames, raw RGB images (1024)

Builds a Kaggle dataset of **lossless full-frame RGB images** that matches the
`*_annotations.json` files created by the paired
[**PhenoBench Multiclass Export — full frames (1024)**](https://www.kaggle.com/code/freimutdiener/export-mc-phenobench-no-partials) notebook.

This is the non-tiled counterpart of `07_export_yoloX_phenobench-tiled.ipynb`;
together the two give COCO-style `images/` + `annotations/` bundles for both
dataset variants.

Attach both inputs before running:

1. `phenobench-raw-dataset-v1-1-0`
2. The output dataset of the TFRecord / COCO export notebook containing
   `train_annotations.json`, `val_annotations.json`, `test_annotations.json`,
   `true_eval_annotations.json`, `dataset_metadata.json`, and related artifacts.

The output layout is:

```text
phenobench_multiclass_full_1024_raw/
├── images/
│   ├── 05-15_00028_P0030852.png
│   ├── ...
├── annotations/
│   ├── train_annotations.json
│   ├── val_annotations.json
│   ├── test_annotations.json
│   ├── true_eval_annotations.json
│   ├── val_test_split.json
│   ├── rep_dataset.json
│   ├── label_map.pbtxt
│   └── dataset_metadata.json
└── raw_images_metadata.json
```

The annotation JSON files use bare file names, so consumers should use
`images/` as their image root.

> **Important:** unlike the tiled variant, nothing is synthesised here. The
> paired exporter wrote COCO entries straight from the on-disk PhenoBench frames
> at their native 1024×1024 size (`export_coco_annotations` reads the sample's
> own dimensions; the `IMAGE_SIZE` knob only affects TFRecords). So the correct
> operation is a **byte-for-byte copy** of the referenced source PNGs — no
> decode, no resize, no re-encode.

## Environment

No `phenobench` / `agri-vision-edge` install is needed: the frames are copied
verbatim from the raw dataset input, and the annotations are copied unchanged
from the paired export. Only the stdlib plus `PIL`/`tqdm` (preinstalled on
Kaggle) are used, which also keeps this notebook immune to loader-version drift.

In [1]:
!python --version

Python 3.10.10


## Configuration

In [2]:
from __future__ import annotations

import json
import shutil
from collections import Counter
from pathlib import Path

from PIL import Image
from tqdm.auto import tqdm


# PhenoBench frames are stored at their native resolution. Nothing is resized
# here; the trainer's resizer downsamples at runtime.
EXPECTED_IMAGE_SIZE = (1024, 1024)

RAW_DATASET_ROOT = Path(
    "/kaggle/input/datasets/freimutdiener/"
    "phenobench-raw-dataset-v1-1-0/PhenoBench"
)

# Change this only to the Kaggle input directory containing the artifacts
# generated by the paired TFRecord / COCO export notebook.
ANNOTATIONS_ROOT = Path(
    "/kaggle/input/datasets/freimutdiener/"
    "mc-phenobench-no-partials"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/phenobench_multiclass_full_1024_raw"
)
IMAGES_ROOT = OUTPUT_ROOT / "images"
COPIED_ARTIFACTS_ROOT = OUTPUT_ROOT / "annotations"

# The paired exporter only ever touched the labelled splits: train frames feed
# train_annotations, and the official val split is halved into val + held-out
# test (true_eval keeps all of it). The unlabelled upstream `test` split is
# deliberately NOT shipped.
SOURCE_SPLITS = ("train", "val")

# Annotation files whose `images[*].file_name` entries must all resolve.
COCO_ARTIFACTS = (
    "train_annotations.json",
    "true_eval_annotations.json",
    "val_annotations.json",
    "test_annotations.json",
)

ANNOTATION_ARTIFACTS = COCO_ARTIFACTS + (
    "label_map.pbtxt",
    "rep_dataset.json",
    "val_test_split.json",
    "dataset_metadata.json",
)

assert RAW_DATASET_ROOT.exists(), RAW_DATASET_ROOT
assert ANNOTATIONS_ROOT.exists(), ANNOTATIONS_ROOT

missing = [name for name in ANNOTATION_ARTIFACTS if not (ANNOTATIONS_ROOT / name).is_file()]
assert not missing, (
    "The annotation input does not look like the output of the paired export "
    f"notebook. Missing: {missing}"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_ROOT.mkdir(parents=True, exist_ok=True)
COPIED_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Raw input:        ", RAW_DATASET_ROOT)
print("Annotations input:", ANNOTATIONS_ROOT)
print("Output:           ", OUTPUT_ROOT)

Raw input:         /kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench
Annotations input: /kaggle/input/datasets/freimutdiener/mc-phenobench-no-partials
Output:            /kaggle/working/phenobench_multiclass_full_1024_raw


## Index the source frames

The output `images/` directory is flat, while upstream keeps one directory per
split. Build a `file_name → source path` index and fail loudly on a cross-split
name collision, which would otherwise silently alias two different frames.

In [3]:
def index_split_images(split: str) -> dict[str, Path]:
    split_images = RAW_DATASET_ROOT / split / "images"
    assert split_images.is_dir(), split_images
    return {
        path.name: path
        for path in sorted(split_images.iterdir())
        if path.is_file()
    }


source_index: dict[str, Path] = {}
source_split: dict[str, str] = {}
collisions: list[str] = []

for split in SOURCE_SPLITS:
    split_index = index_split_images(split)
    for file_name, path in split_index.items():
        if file_name in source_index:
            collisions.append(file_name)
        source_index[file_name] = path
        source_split[file_name] = split
    print(f"{split}: {len(split_index)} frames")

assert not collisions, (
    f"Frame names are not unique across {SOURCE_SPLITS}: {collisions[:10]}"
)

print("Indexed frames total:", len(source_index))

train: 1407 frames
val: 772 frames
Indexed frames total: 2179


## Resolve which frames the annotations reference

Copying is driven by the annotations rather than by the raw directory listing,
so the dataset contains exactly the frames the COCO files point at — no orphan
images, no missing ones.

In [4]:
referenced: dict[str, set[str]] = {}

for artifact in COCO_ARTIFACTS:
    coco = json.loads((ANNOTATIONS_ROOT / artifact).read_text())

    names = [entry["file_name"] for entry in coco["images"]]
    duplicates = sorted(
        name for name, count in Counter(names).items() if count > 1
    )
    assert not duplicates, (
        f"{artifact}: duplicate image file names: {duplicates[:10]}"
    )

    for name in names:
        referenced.setdefault(name, set()).add(artifact)

    print(f"{artifact}: {len(coco['images'])} images, "
          f"{len(coco['annotations'])} annotations")

unresolved = sorted(name for name in referenced if name not in source_index)
assert not unresolved, (
    f"{len(unresolved)} referenced frame(s) are not in the raw input; "
    f"first entries: {unresolved[:10]}"
)

referenced_names = sorted(referenced)
print("\nDistinct referenced frames:", len(referenced_names))
print("Per source split:", dict(sorted(
    Counter(source_split[name] for name in referenced_names).items()
)))

train_annotations.json: 1407 images, 16450 annotations
true_eval_annotations.json: 772 images, 10408 annotations
val_annotations.json: 386 images, 5170 annotations
test_annotations.json: 386 images, 5238 annotations

Distinct referenced frames: 2179
Per source split: {'train': 1407, 'val': 772}


## Copy the full-frame RGB images

`shutil.copy2` preserves the original bytes, so the PNGs stay pixel-identical to
upstream. Existing files are verified rather than silently overwritten with
potentially incompatible content.

In [5]:
def copy_frames(names: list[str]) -> dict:
    written = 0
    reused = 0
    suffixes = Counter()

    for file_name in tqdm(names, desc="Copying frames", unit="image"):
        source = source_index[file_name]
        target = IMAGES_ROOT / file_name

        suffixes[target.suffix.lower()] += 1

        if target.exists():
            if target.stat().st_size != source.stat().st_size:
                raise RuntimeError(
                    f"Existing image differs from the raw source: {target} "
                    f"({target.stat().st_size} != {source.stat().st_size} bytes)"
                )
            reused += 1
            continue

        # Verbatim byte copy: no decode, no resize, no re-encode.
        shutil.copy2(source, target)
        written += 1

    return {
        "frames": len(names),
        "written": written,
        "reused": reused,
        "suffixes": dict(sorted(suffixes.items())),
    }


write_stats = {}
for split in SOURCE_SPLITS:
    split_names = [
        name for name in referenced_names if source_split[name] == split
    ]
    write_stats[split] = copy_frames(split_names)
    print(split, write_stats[split])

Copying frames:   0%|          | 0/1407 [00:00<?, ?image/s]

train {'frames': 1407, 'written': 1407, 'reused': 0, 'suffixes': {'.png': 1407}}


Copying frames:   0%|          | 0/772 [00:00<?, ?image/s]

val {'frames': 772, 'written': 772, 'reused': 0, 'suffixes': {'.png': 772}}


## Copy the paired annotation artifacts

They are copied unchanged, so their image IDs, partial/do-not-care flags, split
indices, and representative-dataset indices retain the meaning established by
the export notebook.

In [6]:
for name in ANNOTATION_ARTIFACTS:
    source = ANNOTATIONS_ROOT / name
    target = COPIED_ARTIFACTS_ROOT / name
    shutil.copy2(source, target)

print("Copied:")
for path in sorted(COPIED_ARTIFACTS_ROOT.iterdir()):
    print(" -", path.name)

Copied:
 - dataset_metadata.json
 - label_map.pbtxt
 - rep_dataset.json
 - test_annotations.json
 - train_annotations.json
 - true_eval_annotations.json
 - val_annotations.json
 - val_test_split.json


## Compatibility verification

This checks every COCO `images[*].file_name` referenced by the copied JSON
against the image directory. It also checks pixel dimensions, which catches an
accidental resize or re-encode, or a raw input that is not the native-resolution
release.

In [7]:
def verify_coco_images(annotation_path: Path) -> dict:
    coco = json.loads(annotation_path.read_text())
    missing = []
    wrong_size = []
    sizes = Counter()

    for entry in tqdm(
        coco["images"],
        desc=f"Checking {annotation_path.name}",
        unit="image",
    ):
        image_path = IMAGES_ROOT / entry["file_name"]

        if not image_path.is_file():
            missing.append(entry["file_name"])
            continue

        with Image.open(image_path) as image:
            actual_size = image.size

        sizes[actual_size] += 1

        expected_size = (int(entry["width"]), int(entry["height"]))
        if actual_size != expected_size:
            wrong_size.append(
                {
                    "file_name": entry["file_name"],
                    "expected": expected_size,
                    "actual": actual_size,
                }
            )

    assert not missing, (
        f"{annotation_path.name}: {len(missing)} referenced image(s) are missing; "
        f"first entries: {missing[:10]}"
    )
    assert not wrong_size, (
        f"{annotation_path.name}: {len(wrong_size)} image dimension mismatch(es); "
        f"first entries: {wrong_size[:3]}"
    )
    assert set(sizes) == {EXPECTED_IMAGE_SIZE}, (
        f"{annotation_path.name}: unexpected frame geometry: {dict(sizes)}"
    )

    return {
        "annotation_file": annotation_path.name,
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "all_images_present": True,
        "all_dimensions_match": True,
    }


verification = [
    verify_coco_images(COPIED_ARTIFACTS_ROOT / name)
    for name in COCO_ARTIFACTS
]

for result in verification:
    print(result)

Checking train_annotations.json:   0%|          | 0/1407 [00:00<?, ?image/s]

Checking true_eval_annotations.json:   0%|          | 0/772 [00:00<?, ?image/s]

Checking val_annotations.json:   0%|          | 0/386 [00:00<?, ?image/s]

Checking test_annotations.json:   0%|          | 0/386 [00:00<?, ?image/s]

{'annotation_file': 'train_annotations.json', 'images': 1407, 'annotations': 16450, 'all_images_present': True, 'all_dimensions_match': True}
{'annotation_file': 'true_eval_annotations.json', 'images': 772, 'annotations': 10408, 'all_images_present': True, 'all_dimensions_match': True}
{'annotation_file': 'val_annotations.json', 'images': 386, 'annotations': 5170, 'all_images_present': True, 'all_dimensions_match': True}
{'annotation_file': 'test_annotations.json', 'images': 386, 'annotations': 5238, 'all_images_present': True, 'all_dimensions_match': True}


## Dataset metadata and final check

In [8]:
image_paths = sorted(
    path for path in IMAGES_ROOT.iterdir()
    if path.is_file()
)

orphans = sorted(
    path.name for path in image_paths if path.name not in referenced
)
assert not orphans, (
    f"{len(orphans)} image(s) are not referenced by any annotation file; "
    f"first entries: {orphans[:10]}"
)

metadata = {
    "source_dataset": "PhenoBench v1.1.0 raw RGB images",
    "source_annotation_artifacts": str(ANNOTATIONS_ROOT),
    "image_root": "images",
    "annotation_root": "annotations",
    "tiling": None,
    "frames": {
        "geometry": list(EXPECTED_IMAGE_SIZE),
        "copy_mode": "verbatim byte copy of the source PNGs",
        "filename_pattern": "<source_file_name> (unchanged)",
        "source_splits": list(SOURCE_SPLITS),
    },
    "splits_written": {
        "train": write_stats["train"],
        # val is deliberately complete because true_eval contains all original
        # validation frames and val/test are subsets of it.
        "val_source": write_stats["val"],
    },
    "images_written_total": len(image_paths),
    "coco_compatibility": verification,
}

(OUTPUT_ROOT / "raw_images_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)

assert image_paths, "No images were written."
assert len(image_paths) == len(referenced_names)
assert (OUTPUT_ROOT / "raw_images_metadata.json").is_file()

print(json.dumps(metadata, indent=2))
print("\nDataset ready:")
print(OUTPUT_ROOT)

{
  "source_dataset": "PhenoBench v1.1.0 raw RGB images",
  "source_annotation_artifacts": "/kaggle/input/datasets/freimutdiener/mc-phenobench-no-partials",
  "image_root": "images",
  "annotation_root": "annotations",
  "tiling": null,
  "frames": {
    "geometry": [
      1024,
      1024
    ],
    "copy_mode": "verbatim byte copy of the source PNGs",
    "filename_pattern": "<source_file_name> (unchanged)",
    "source_splits": [
      "train",
      "val"
    ]
  },
  "splits_written": {
    "train": {
      "frames": 1407,
      "written": 1407,
      "reused": 0,
      "suffixes": {
        ".png": 1407
      }
    },
    "val_source": {
      "frames": 772,
      "written": 772,
      "reused": 0,
      "suffixes": {
        ".png": 772
      }
    }
  },
  "images_written_total": 2179,
  "coco_compatibility": [
    {
      "annotation_file": "train_annotations.json",
      "images": 1407,
      "annotations": 16450,
      "all_images_present": true,
      "all_dimensions_match